In [8]:
import pandas as pd
import numpy as np
import os



In [ ]:
data_path = "../data/raw/"

In [9]:

customers = pd.read_csv(
    data_path + "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    data_path + "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    data_path + "olist_order_items_dataset.csv"
)

order_payments = pd.read_csv(
    data_path + "olist_order_payments_dataset.csv"
)

order_reviews = pd.read_csv(
    data_path + "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    data_path + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    data_path + "olist_sellers_dataset.csv"
)

product_category_translation = pd.read_csv(
    data_path + "product_category_name_translation.csv"
)

geolocation = pd.read_csv(
    data_path + "olist_geolocation_dataset.csv"
)

In [10]:

tables = {
    "CUSTOMERS": customers,
    "ORDERS": orders,
    "ORDER_ITEMS": order_items,
    "ORDER_PAYMENTS": order_payments,
    "ORDER_REVIEWS": order_reviews,
    "PRODUCTS": products,
    "SELLERS": sellers,
    "PRODUCT_CATEGORY_TRANSLATION": product_category_translation,
    "GEOLOCATION": geolocation
}

In [11]:
cleaned_tables = {
    table_name: df.copy()
    for table_name, df in tables.items()
}

In [12]:

customers_clean = cleaned_tables["CUSTOMERS"]
orders_clean = cleaned_tables["ORDERS"]
order_items_clean = cleaned_tables["ORDER_ITEMS"]
order_payments_clean = cleaned_tables["ORDER_PAYMENTS"]
order_reviews_clean = cleaned_tables["ORDER_REVIEWS"]
products_clean = cleaned_tables["PRODUCTS"]
sellers_clean = cleaned_tables["SELLERS"]
product_category_translation_clean = cleaned_tables["PRODUCT_CATEGORY_TRANSLATION"]
geolocation_clean = cleaned_tables["GEOLOCATION"]

In [13]:
tables

{'CUSTOMERS':                             customer_id                customer_unique_id  \
 0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 ...                                 ...                               ...   
 99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
 99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
 99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
 99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
 99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   
 
        customer_zip_code_prefix          custome

In [14]:
missing_before = []

for table_name, df in cleaned_tables.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_before.append({
                "Table": table_name,
                "Column": column,
                "Missing_Count": missing_count,
                "Missing_Percentage": round(
                    (missing_count / len(df)) * 100, 2
                )
            })

missing_before_df = pd.DataFrame(missing_before)

missing_before_df

,Table,Column,Missing_Count,Missing_Percentage
0,ORDERS,order_approved_at,160,0.16
1,ORDERS,order_delivered_carrier_date,1783,1.79
2,ORDERS,order_delivered_customer_date,2965,2.98
3,ORDER_REVIEWS,review_comment_title,87656,88.34
4,ORDER_REVIEWS,review_comment_message,58247,58.70
5,PRODUCTS,product_category_name,610,1.85
6,PRODUCTS,product_name_lenght,610,1.85
7,PRODUCTS,product_description_lenght,610,1.85
8,PRODUCTS,product_photos_qty,610,1.85
9,PRODUCTS,product_weight_g,2,0.01


In [15]:
orders_clean[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].isna().sum()

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

In [16]:
order_reviews_clean["review_comment_title"] = (
    order_reviews_clean["review_comment_title"]
    .fillna("")
)

order_reviews_clean["review_comment_message"] = (
    order_reviews_clean["review_comment_message"]
    .fillna("")
)
print(
    "Missing review titles:",
    order_reviews_clean["review_comment_title"].isna().sum()
)

print(
    "Missing review messages:",
    order_reviews_clean["review_comment_message"].isna().sum()
)

Missing review titles: 0
Missing review messages: 0


In [17]:
products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

In [18]:
products_clean["product_category_name"].isna().sum()

np.int64(0)

In [19]:
product_numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_clean[product_numeric_columns].isna().sum()

product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [20]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )
orders_clean[order_date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [21]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

In [23]:
order_reviews_clean["review_creation_date"] = pd.to_datetime(
    order_reviews_clean["review_creation_date"],
    errors="coerce"
)
order_reviews_clean["review_creation_date"].dtype

dtype('<M8[us]')

In [25]:
numeric_columns = {
    "ORDER_ITEMS": [
        "order_item_id",
        "price",
        "freight_value"
    ],
    "ORDER_PAYMENTS": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],
    "ORDER_REVIEWS": [
        "review_score"
    ],
    "PRODUCTS": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for column in numeric_columns["ORDER_ITEMS"]:
    order_items_clean[column] = pd.to_numeric(
        order_items_clean[column],
        errors="coerce"
    )

for column in numeric_columns["ORDER_PAYMENTS"]:
    order_payments_clean[column] = pd.to_numeric(
        order_payments_clean[column],
        errors="coerce"
    )

order_reviews_clean["review_score"] = pd.to_numeric(
    order_reviews_clean["review_score"],
    errors="coerce"
)

for column in numeric_columns["PRODUCTS"]:
    products_clean[column] = pd.to_numeric(
        products_clean[column],
        errors="coerce"
    )

In [26]:
print("ORDER_ITEMS")
print(order_items_clean[numeric_columns["ORDER_ITEMS"]].dtypes)

print("\nORDER_PAYMENTS")
print(order_payments_clean[numeric_columns["ORDER_PAYMENTS"]].dtypes)

print("\nORDER_REVIEWS")
print(order_reviews_clean[["review_score"]].dtypes)

print("\nPRODUCTS")
print(products_clean[numeric_columns["PRODUCTS"]].dtypes)

ORDER_ITEMS
order_item_id      int64
price            float64
freight_value    float64
dtype: object

ORDER_PAYMENTS
payment_sequential        int64
payment_installments      int64
payment_value           float64
dtype: object

ORDER_REVIEWS
review_score    int64
dtype: object

PRODUCTS
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [27]:
orders_clean["order_status"] = (
    orders_clean["order_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [28]:
orders_clean["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64[pyarrow]

In [29]:
valid_order_status = {
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved"
}

invalid_order_status = orders_clean[
    ~orders_clean["order_status"].isin(valid_order_status)
]

print("Invalid order status records:", len(invalid_order_status))

Invalid order status records: 0


In [31]:
order_payments_clean["payment_type"] = (
    order_payments_clean["payment_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)
order_payments_clean["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64[pyarrow]

In [32]:
customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .astype("string")
    .str.strip()
)

customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [33]:
sellers_clean["seller_city"] = (
    sellers_clean["seller_city"]
    .astype("string")
    .str.strip()
)

sellers_clean["seller_state"] = (
    sellers_clean["seller_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [34]:
products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [35]:
product_category_translation_clean[
    "product_category_name"
] = (
    product_category_translation_clean["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

product_category_translation_clean[
    "product_category_name_english"
] = (
    product_category_translation_clean[
        "product_category_name_english"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [36]:
invalid_review_scores = order_reviews_clean[
    ~order_reviews_clean["review_score"].isin([1, 2, 3, 4, 5])
]

print("Invalid review scores:", len(invalid_review_scores))

Invalid review scores: 0


In [37]:
invalid_freight = order_items_clean[
    order_items_clean["freight_value"] < 0
]

print("Invalid freight records:", len(invalid_freight))

Invalid freight records: 0


In [38]:
invalid_installments = order_payments_clean[
    order_payments_clean["payment_installments"] <= 0
]

print("Invalid payment installment records:", len(invalid_installments))

Invalid payment installment records: 2


In [39]:
cleaned_tables = {
    "CUSTOMERS": customers_clean,
    "ORDERS": orders_clean,
    "ORDER_ITEMS": order_items_clean,
    "ORDER_PAYMENTS": order_payments_clean,
    "ORDER_REVIEWS": order_reviews_clean,
    "PRODUCTS": products_clean,
    "SELLERS": sellers_clean,
    "PRODUCT_CATEGORY_TRANSLATION": product_category_translation_clean,
    "GEOLOCATION": geolocation_clean
}

In [40]:
geo_before = len(geolocation_clean)

geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

geo_after = len(geolocation_clean)

print("GEOLOCATION rows before:", geo_before)
print("GEOLOCATION rows after:", geo_after)
print("Exact duplicate rows removed:", geo_before - geo_after)

GEOLOCATION rows before: 1000163
GEOLOCATION rows after: 738332
Exact duplicate rows removed: 261831


In [41]:
cleaned_tables["GEOLOCATION"] = geolocation_clean

In [42]:
order_item_duplicates = order_items_clean.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print(
    "Duplicate ORDER_ITEMS business keys:",
    order_item_duplicates
)

Duplicate ORDER_ITEMS business keys: 0


In [44]:
review_id_duplicates = order_reviews_clean[
    order_reviews_clean["review_id"].duplicated(keep=False)
].sort_values("review_id")

review_id_duplicates.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,,,2017-09-21,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,,,2017-09-21,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,,,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,,,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [45]:
review_business_duplicates = order_reviews_clean.duplicated(
    subset=["review_id", "order_id"]
).sum()

print(
    "Duplicate REVIEW_ID + ORDER_ID combinations:",
    review_business_duplicates
)

Duplicate REVIEW_ID + ORDER_ID combinations: 0


In [46]:
geo_key_counts = (
    geolocation_clean["geolocation_zip_code_prefix"]
    .value_counts()
)

geo_key_counts.head(10)

geolocation_zip_code_prefix
38400    779
35500    751
11680    727
11740    678
36400    627
38408    621
39400    620
35162    611
37200    596
35900    589
Name: count, dtype: int64

In [47]:
row_count_summary = []

for table_name in tables:
    original_rows = len(tables[table_name])
    cleaned_rows = len(cleaned_tables[table_name])

    row_count_summary.append({
        "Table": table_name,
        "Original_Rows": original_rows,
        "Cleaned_Rows": cleaned_rows,
        "Rows_Removed": original_rows - cleaned_rows
    })

row_count_df = pd.DataFrame(row_count_summary)

row_count_df

,Table,Original_Rows,Cleaned_Rows,Rows_Removed
0,CUSTOMERS,99441,99441,0
1,ORDERS,99441,99441,0
2,ORDER_ITEMS,112650,112650,0
3,ORDER_PAYMENTS,103886,103886,0
4,ORDER_REVIEWS,99224,99224,0
5,PRODUCTS,32951,32951,0
6,SELLERS,3095,3095,0
7,PRODUCT_CATEGORY_TRANSLATION,71,71,0
8,GEOLOCATION,1000163,738332,261831


In [48]:
final_missing = []

for table_name, df in cleaned_tables.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            final_missing.append({
                "Table": table_name,
                "Column": column,
                "Missing_Count": missing_count,
                "Missing_Percentage": round(
                    (missing_count / len(df)) * 100,
                    2
                )
            })

final_missing_df = pd.DataFrame(final_missing)

final_missing_df

,Table,Column,Missing_Count,Missing_Percentage
0,ORDERS,order_approved_at,160,0.16
1,ORDERS,order_delivered_carrier_date,1783,1.79
2,ORDERS,order_delivered_customer_date,2965,2.98
3,PRODUCTS,product_name_lenght,610,1.85
4,PRODUCTS,product_description_lenght,610,1.85
5,PRODUCTS,product_photos_qty,610,1.85
6,PRODUCTS,product_weight_g,2,0.01
7,PRODUCTS,product_length_cm,2,0.01
8,PRODUCTS,product_height_cm,2,0.01
9,PRODUCTS,product_width_cm,2,0.01


In [49]:
for table_name, df in cleaned_tables.items():

    print("=" * 70)
    print(table_name)
    print("=" * 70)

    print(df.dtypes)
    print()

CUSTOMERS
customer_id                    str
customer_unique_id             str
customer_zip_code_prefix     int64
customer_city               string
customer_state              string
dtype: object

ORDERS
order_id                                    str
customer_id                                 str
order_status                             string
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

ORDER_ITEMS
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

ORDER_PAYMENTS
order_id                    str
payment_sequential        int64
payment_type          

In [50]:
final_quality_summary = []

for table_name, df in cleaned_tables.items():

    final_quality_summary.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Total_Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum())
    })

final_quality_df = pd.DataFrame(final_quality_summary)

final_quality_df

,Table,Rows,Columns,Total_Missing_Values,Duplicate_Rows
0,CUSTOMERS,99441,5,0,0
1,ORDERS,99441,8,4908,0
2,ORDER_ITEMS,112650,7,0,0
3,ORDER_PAYMENTS,103886,5,0,0
4,ORDER_REVIEWS,99224,7,0,0
5,PRODUCTS,32951,9,1838,0
6,SELLERS,3095,4,0,0
7,PRODUCT_CATEGORY_TRANSLATION,71,2,0,0
8,GEOLOCATION,738332,5,0,0


In [51]:
cleaned_path = "../data/cleaned/"

os.makedirs(cleaned_path, exist_ok=True)


In [52]:
output_files = {
    "CUSTOMERS": "customers_cleaned.csv",
    "ORDERS": "orders_cleaned.csv",
    "ORDER_ITEMS": "order_items_cleaned.csv",
    "ORDER_PAYMENTS": "order_payments_cleaned.csv",
    "ORDER_REVIEWS": "order_reviews_cleaned.csv",
    "PRODUCTS": "products_cleaned.csv",
    "SELLERS": "sellers_cleaned.csv",
    "PRODUCT_CATEGORY_TRANSLATION": "product_category_translation_cleaned.csv",
    "GEOLOCATION": "geolocation_cleaned.csv"
}

for table_name, df in cleaned_tables.items():

    file_path = os.path.join(
        cleaned_path,
        output_files[table_name]
    )

    df.to_csv(file_path, index=False)

    print(f"Saved: {file_path}")

Saved: ../data/cleaned/customers_cleaned.csv
Saved: ../data/cleaned/orders_cleaned.csv
Saved: ../data/cleaned/order_items_cleaned.csv
Saved: ../data/cleaned/order_payments_cleaned.csv
Saved: ../data/cleaned/order_reviews_cleaned.csv
Saved: ../data/cleaned/products_cleaned.csv
Saved: ../data/cleaned/sellers_cleaned.csv
Saved: ../data/cleaned/product_category_translation_cleaned.csv
Saved: ../data/cleaned/geolocation_cleaned.csv
